# 🐦 Twitter Sentiment Analysis — Exploratory Notebook

This notebook walks through the full pipeline:
1. Load and explore sample tweet data
2. Clean and preprocess text
3. Run NLTK VADER sentiment analysis
4. Visualise results with charts
5. (Optional) Fine-tune with SpaCy


In [ ]:
# Install dependencies (run once)
# !pip install nltk spacy pandas matplotlib seaborn wordcloud
# !python -m spacy download en_core_web_sm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Ensure NLTK data
for resource in ['vader_lexicon', 'punkt', 'stopwords']:
    nltk.download(resource, quiet=True)

print('✅ All imports successful!')

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/sample_tweets.csv')
print(f'Loaded {len(df)} tweets')
df.head()

## 2. Preprocess Text

In [ ]:
import re

def clean_tweet(text):
    text = re.sub(r'http\S+|www\.\S+', '', text)   # Remove URLs
    text = re.sub(r'@\w+', '', text)               # Remove @mentions
    text = re.sub(r'#(\w+)', r'\1', text)          # Keep hashtag words
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_tweet)
df[['text', 'clean_text']].head()

## 3. VADER Sentiment Analysis

In [ ]:
sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    scores = sia.polarity_scores(text)
    compound = scores['compound']
    if compound >= 0.05:
        return 'positive', compound
    elif compound <= -0.05:
        return 'negative', compound
    else:
        return 'neutral', compound

df[['sentiment', 'compound']] = df['clean_text'].apply(
    lambda t: pd.Series(get_sentiment(t))
)

print(df['sentiment'].value_counts())
df[['text', 'sentiment', 'compound']].head(10)

## 4. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Twitter Sentiment Analysis Results', fontsize=16, fontweight='bold')

colors = {'positive': '#22c55e', 'negative': '#ef4444', 'neutral': '#f59e0b'}
counts = df['sentiment'].value_counts()

# Pie chart
axes[0].pie(
    counts.values, labels=counts.index,
    colors=[colors[l] for l in counts.index],
    autopct='%1.1f%%', startangle=90
)
axes[0].set_title('Sentiment Distribution')

# Bar chart
bar_colors = [colors[l] for l in counts.index]
axes[1].bar(counts.index, counts.values, color=bar_colors, edgecolor='white', linewidth=0.8)
axes[1].set_title('Tweet Counts by Sentiment')
axes[1].set_ylabel('Number of Tweets')

# Compound score histogram
axes[2].hist(df['compound'], bins=20, color='#6366f1', edgecolor='white', linewidth=0.5)
axes[2].axvline(x=0.05,  color='#22c55e', linestyle='--', label='Positive threshold')
axes[2].axvline(x=-0.05, color='#ef4444', linestyle='--', label='Negative threshold')
axes[2].set_title('Compound Score Distribution')
axes[2].set_xlabel('Compound Score')
axes[2].legend()

plt.tight_layout()
plt.savefig('../data/sentiment_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to data/sentiment_results.png')

## 5. Export Results

In [ ]:
df.to_csv('../data/tweets_with_sentiment.csv', index=False)
print('Results saved to data/tweets_with_sentiment.csv')

summary = df.groupby('sentiment').agg(
    count=('compound', 'count'),
    avg_compound=('compound', 'mean'),
    avg_likes=('likes', 'mean'),
).round(3)
print('\n--- Summary ---')
print(summary)